In [5]:
import torch
import torchvision
import torchvision.transforms.v2

transforms = torchvision.transforms.v2.Compose([
    torchvision.transforms.v2.Grayscale(num_output_channels=3),
    torchvision.models.ViT_B_32_Weights.DEFAULT.transforms()
])
train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)

train_y = train_ds.targets
test_y = test_ds.targets

In [6]:
import zigzag.utils
from zigzag.pipelines.validate import train_validate
from zigzag.pipelines.validate import validate_pretrained

dumper = zigzag.utils.UniversalDumper(f"ablation_results/mnist/vit_b_32")

In [7]:
model = torchvision.models.vit_b_32(
    num_classes=1000, weights=torchvision.models.ViT_B_32_Weights.DEFAULT
)
model.heads = torch.nn.Identity()
pretrained_dumper = dumper.make_subdumper("pretrained")
validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)

Got the result from ablation_results/mnist/vit_b_32/pretrained\train_embeddings.pt
Got the result from ablation_results/mnist/vit_b_32/pretrained\test_embeddings.pt
Got the result from ablation_results/mnist/vit_b_32/pretrained\train_head_history.csv


d:\HSE\CourseProject-Mag-1\zigzag\utils\dumper.py:116: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(file)


In [8]:
finetuned_dumper = dumper.make_subdumper("finetuned")
model = torchvision.models.vit_b_32(
    num_classes=1000, weights=torchvision.models.ViT_B_32_Weights.DEFAULT
)
model.heads = pretrained_dumper.get_dump("trained_head")
train_validate(model, train_ds, test_ds, finetuned_dumper, learning_rate=1e-5)

Got the result from ablation_results/mnist/vit_b_32/pretrained\trained_head.pth


Training: 100%|██████████| 10/10 [1:46:23<00:00, 638.31s/it, Accuracy=0.994, AUC-ROC=1, Precision=0.994, Recall=0.994, F1-score=0.994, TOP-2 Accuracy=0.998, TOP-3 Accuracy=1, TOP-5 Accuracy=1, TOP-7 Accuracy=1, TOP-9 Accuracy=1]   


Saving the result to ablation_results/mnist/vit_b_32/finetuned\train_model_history.csv
Saving the result to ablation_results/mnist/vit_b_32/finetuned\trained_model.pth
